# Leaderboard Pipeline 05 — Duplicate-Aware Stratified Folds

This notebook builds five folds from the reviewed clean manifest using `StratifiedGroupKFold`.

Exact duplicates and manually confirmed DINO duplicate groups stay in the same fold. Every image without a strict duplicate group receives its own unique group.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "BDC2026"
CLEAN_ROOT = REPO_ROOT / "BDC2026_clean_v1"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CLEAN_ROOT:", CLEAN_ROOT)


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

STAGE_DIR = PIPELINE_ROOT / "05_folds"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_MANIFEST_PATH = PIPELINE_ROOT / "04_clean_dataset" / "clean_manifest.csv"
DUPLICATE_PATH = PIPELINE_ROOT / "03_duplicates" / "oof_with_duplicate_groups.csv"

clean = pd.read_csv(CLEAN_MANIFEST_PATH)
duplicates = pd.read_csv(DUPLICATE_PATH)
print("Clean images:", len(clean), "Duplicate map rows:", len(duplicates))


In [ ]:
strict_map = (
    duplicates.drop_duplicates("path_key")
    .set_index("path_key")["strict_group_id"]
    .to_dict()
)
clean["group_id"] = clean["source_path_key"].map(strict_map)
missing_group = clean["group_id"].isna()
clean.loc[missing_group, "group_id"] = "UNIQUE_" + clean.loc[missing_group, "source_path_key"]
clean["label"] = clean["final_label"].astype(int)
clean["class_name"] = clean["final_class_name"]
clean["path_key"] = clean["clean_path_key"]

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
clean["fold"] = -1
for fold, (_, valid_index) in enumerate(
    splitter.split(clean, clean["label"], groups=clean["group_id"])
):
    clean.loc[valid_index, "fold"] = fold

assert (clean["fold"] >= 0).all()
clean["fold"] = clean["fold"].astype(int)


In [ ]:
leak_check = clean.groupby("group_id")["fold"].nunique()
assert leak_check.max() == 1, leak_check[leak_check > 1]

class_distribution = clean.groupby(["fold", "class_name"]).size().unstack(fill_value=0)
fold_sizes = clean.groupby("fold").size().rename("images")
group_distribution = clean.groupby("fold")["group_id"].nunique().rename("groups")
display(class_distribution)
display(pd.concat([fold_sizes, group_distribution], axis=1))


In [ ]:
fold_columns = [
    "source_path", "source_path_key", "clean_relative_path", "path_key",
    "class_name", "label", "group_id", "fold",
]
folds = clean[fold_columns].copy()
fold_path = STAGE_DIR / "train_folds_duplicate_aware.csv"
folds.to_csv(fold_path, index=False)
class_distribution.to_csv(STAGE_DIR / "fold_class_distribution.csv")
pd.concat([fold_sizes, group_distribution], axis=1).to_csv(
    STAGE_DIR / "fold_group_distribution.csv"
)

summary = {
    "images": len(folds),
    "folds": 5,
    "groups": int(folds["group_id"].nunique()),
    "largest_group": int(folds.groupby("group_id").size().max()),
    "cross_fold_group_leakage": int(
        (folds.groupby("group_id")["fold"].nunique() > 1).sum()
    ),
}
(STAGE_DIR / "fold_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
summary


In [ ]:
command = (
    "python scripts/train_with_precomputed_folds.py "
    f"--data-root {CLEAN_ROOT} "
    f"--fold-csv {STAGE_DIR / 'train_folds_duplicate_aware.csv'} "
    "--output-dir ./experiments/E01_clean384_seed42 "
    "--image-size 384 --epochs 25 --batch-size 2 --valid-batch-size 4 "
    "--grad-accum 8 --seed 42 --use-class-weights"
)
print(command)


## Why this matters

Normal stratified folds can place duplicate copies in different folds and inflate OOF performance. These folds keep strict duplicate groups together, producing a more realistic validation estimate for deciding whether S02 is worth using.
